<a href="https://colab.research.google.com/github/hknypaul/Fan-Base/blob/master/Image_Playground_DeFooocus_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title DeFooocus
#@markdown **Launch the interface DeFocus (Fooocus fork)** | You need to connect with T4/A100/V100
#@markdown ****
#@markdown *Attention!* When working in the interface with the FaceSwap and CPDS controlnet, crashes are possible; it is also recommended to work in *Extreme speed* mode for additional stability. When working with the ImagePrompt and PyraCanny controls, 85% of the work will be stable.
#@markdown ****

print("[DeFooocus] Preparing ...")

theme = "dark" #@param ["dark", "light"]
preset = "playground_v2.5" #@param ["deafult", "realistic", "anime", "lcm", "sai", "turbo", "lighting", "hypersd", "playground_v2.5", "dpo", "spo", "sd1.5"]
advenced_args = "--share --attention-split --always-high-vram --disable-offload-from-vram --all-in-fp16" #@param {type: "string"}

if preset != "deafult":
  args = f"{advenced_args} --theme {theme} --preset {preset}"
else:
  args = f"{advenced_args} --theme {theme}"

!pip install -q pygit2==1.12.2
%cd /content
!git clone https://github.com/ehristoforu/DeFooocus.git
%cd /content/DeFooocus
!pip install -q -r requirements_versions.txt

print("[DeFooocus] Starting ...")
!python entry_with_update.py $args

In [ ]:
# @title
# 🖼️ DeFooocus - Image-to-Image with Interactive Picker
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests
from PIL import Image
from io import BytesIO
import base64
from google.colab import files

print("🎨 DeFooocus Image-to-Image Generator")

# Widgets
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Choose Image'
)

prompt_input = widgets.Textarea(
    value='a fantasy castle on a hill, dramatic lighting, artstation',
    placeholder='Describe your image...',
    description='Prompt:',
    layout=widgets.Layout(width='100%', height='80px')
)

neg_prompt_input = widgets.Textarea(
    value='blurry, cartoon, 3d, disfigured',
    placeholder='Things to avoid...',
    description='Negative Prompt:',
    layout=widgets.Layout(width='100%', height='60px')
)

variation_dropdown = widgets.Dropdown(
    options=['Vary (Subtle)', 'Vary (Strong)', 'Upscale (2x)'],
    value='Vary (Subtle)',
    description='Variation:'
)

performance_dropdown = widgets.Dropdown(
    options=['Speed', 'Quality'],
    value='Speed',
    description='Performance:'
)

aspect_dropdown = widgets.Dropdown(
    options=[
        '1152*896', '1216*832', '1344*768',
        '1536*640', '640*1536', '768*1344',
        '896*1152', '1024*1024'
    ],
    value='1152*896',
    description='Aspect Ratio:'
)

run_button = widgets.Button(label="🚀 Generate", button_style="success")
output_area = widgets.Output()

# Display UI
display(widgets.VBox([
    uploader,
    prompt_input,
    neg_prompt_input,
    variation_dropdown,
    performance_dropdown,
    aspect_dropdown,
    run_button
]))
display(output_area)

# On click: Generate
def on_run_click(b):
    with output_area:
        clear_output()
        if len(uploader.data) == 0:
            print("❗ Please upload an image first.")
            return

        # Load image
        name = uploader.filename
        data = uploader.data[0]
        print(f"🖼️ Loading {name}...")

        img = Image.open(BytesIO(data)).convert('RGB')
        # Show preview
        buffer = BytesIO()
        img.save(buffer, format="PNG")
        img_str = base64.b64encode(buffer.getvalue()).decode()

        display(widgets.HTML("<h4>🖼️ Input Image:</h4>"))
        display(img)

        # Build payload
        print("🚀 Sending img2img request...")
        payload = {
            "prompt": prompt_input.value,
            "negative_prompt": neg_prompt_input.value,
            "style_selections": [],
            "image_number": 1,
            "image": img_str,
            "image_styles": ["Enhance"],
            "performance_selection": performance_dropdown.value,
            "aspect_ratios_selection": aspect_dropdown.value,
            "advanced_checkbox": True,
            "max_image_number": 1,
            "mixing_image_prompt_and_vary_upscale": False,
            "mixing_image_prompt_and_inpaint": False,
            "super_resolution": True,
            "uov_input_image": img_str,
            "uov_method": variation_dropdown.value,
            "outpaint_selections": [],
            "outpaint_distance_left": 0,
            "outpaint_distance_right": 0,
            "outpaint_distance_top": 0,
            "outpaint_distance_bottom": 0,
        }

        try:
            response = requests.post("http://127.0.0.1:7860/v1/generation/", json=payload, timeout=300)
            if response.status_code == 200:
                result = response.json()
                display(widgets.HTML("<h4>🎨 Output Image(s):</h4>"))
                for i, img_data in enumerate(result['images']):
                    image = Image.open(BytesIO(base64.b64decode(img_data)))
                    image.save(f"img2img_output_{i}.png")
                    display(image)
                # Auto-download
                files.download("img2img_output_0.png")
            else:
                print("❌ Error:", response.status_code)
                print(response.text)
        except Exception as e:
            print("💥 Request failed:", str(e))
            print("💡 Make sure the DeFooocus backend is running.")

run_button.on_click(on_run_click)